# Phase 8 — Streamlit Dashboard Demo

Notebook 08. `dashboard/streamlit_app.py` is a 3-page Streamlit app:

1. **📊 Business Dashboard** — KPIs + category/pricing/rating/discount charts + correlation heatmap
2. **🔍 Review Explorer** — filter products, search review text, inspect a product's reviews + sentiment
3. **🎯 Recommendation** — pick a product, get TF-IDF content-based recommendations

Run it with:
```bash
streamlit run dashboard/streamlit_app.py
```

This notebook doesn't run the Streamlit server itself (that needs a terminal,
not a notebook kernel) — instead it exercises the same underlying data-logic
functions the dashboard calls (`filter_products`, `compute_kpis`,
`sentiment_distribution`), so you can sanity-check the dashboard's behavior
inline, and previews the same charts each dashboard page renders.


In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline


## Underlying data logic (same functions the dashboard calls)

In [ ]:
from dashboard.streamlit_app import (
    load_data, filter_products, compute_kpis, sentiment_distribution, get_category_options,
)

df = load_data()
df.shape


In [ ]:
# Same filters as the sidebar widgets in the app
categories = get_category_options(df)[:2]
filtered = filter_products(df, categories=categories, price_range=(0, 5000), rating_range=(4.0, 5.0))
print(f"{len(filtered)} products match: category in {categories}, price <= 5000, rating >= 4.0")
compute_kpis(filtered)


In [ ]:
sentiment_distribution(filtered)


## Preview: Business Dashboard charts

In [ ]:
from src.visualization import plot_bar, plot_hist, plot_scatter

counts = filtered["main_category"].value_counts()
_ = plot_bar(counts.index, counts.values, "Products by Category (filtered)", save=False)
_ = plot_hist(filtered["rating"], "Rating Distribution (filtered)", xlabel="Rating", bins=15, save=False)
_ = plot_scatter(filtered, x="discount_percentage", y="rating", title="Discount % vs. Rating (filtered)", save=False)


## Preview: Recommendation page

In [ ]:
from dashboard.streamlit_app import get_product_options
from src.recommendation import TfidfRecommender

recommender = TfidfRecommender(df)
example = get_product_options(df)[0]
query = " ".join(example.split()[:4])
print("Query:", query)
recommender.recommend(product_name=query, top_k=5)


## Notes for running the live app

- The dashboard caches data loading (`st.cache_data`) and the recommender
  (`st.cache_resource`) so filtering/recommending stays snappy after first load.
- Sentiment uses the same fallback lexicon as `src/sentiment.py` in
  environments without `vaderSentiment` installed — install it for real
  VADER scores, no dashboard code changes needed.
- Review Explorer operates at the *product* level (one row = one product,
  with all of that product's scraped reviews concatenated) rather than one
  row per individual review, matching this dataset's native structure.